### Generating montage of RGB images from hiPSC-CMs
#### Lavie Ngo
#### This notebook: Generates one montage per condition comparing control, 1 week of maturation media, and 2 weeks of maturation media, after CARBONITE imaging.

#### Experiment info:
- Differentiated vCMs (D0: 08/31/2026) were replated on D18 (09/18/26)into two 96-well plates and cultured in maintenance media for 1 week until D25 (09/25/26). On D25, 8 wells from each plate were swapped to Feyen maturation media. The other 8 wells continued to be cultured in maintenance media.
    - Plate 1:
        - Wells B02-B09: Maintenance media (control)
        - Wells C02-C09: Feyen maturation media (treatment) for 1 week; CARBONITE done on 10/02/26
        - Plate fixed after CARBONITE.
    - Plate 2:
        - Wells B02-B09: Maintenance media (control)
        - Wells C02-C09: Feyen maturation media (treatment) for 2 weeks; CARBONITE done on 10/09/26
    - Both plates were stained for MYL2, ANP, cTnT, and DAPI and imaged on 10/09/26.

In [ ]:
# Reusable montage function
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import tifffile

TIFF_PATH = Path(
    "/Volumes/SaucermanLab/Lavie/LRP1/Experiments/IRI-experiments/"
    "hiPSC CMs/LLN-20260925-WT11-vCM-E1-D25/20260925-allchannels-merged.tif"
)

CHANNEL_CONFIG = {
    "w1": {"name": "Hoechst", "color": (0.0, 1.0, 1.0)},  # cyan
    "w3": {"name": "A5",      "color": (1.0, 0.0, 1.0)},  # magenta
    "w4": {"name": "PI",      "color": (1.0, 1.0, 0.0)},  # yellow
}


# Explicit well IDs match the 20260916 experiment layout.
PLATE_CONDITIONS = {
    "norm": {
        "normoxia": ["B02", "C02", "D02", "B03", "C03", "D03"],
        "ischemia buffer only": ["B04", "C04", "D04"],
        "ischemia buffer only + reperfusion": ["B05", "C05", "D05"],
        "PI3K inhibitor": ["B06", "C06", "D06", "B07", "C07", "D07"],
        "Src inhibitor": ["B08", "C08", "D08", "B09", "C09", "D09"],
    },
    "IR": {
        "IR": ["B02", "C02", "D02"],
        "SP34": ["B03", "C03", "D03"],
        "DMSO": ["B04", "C04", "D04"],
        "IR + SP16": ["B05", "C05", "D05"],
        "IR + SP16 + PI3K inhibitor": ["B06", "C06", "D06"],
        "IR + SP16 + Src inhibitor": ["B07", "C07", "D07"],
        "IR + PI3K inhibitor": ["B08", "C08", "D08"],
        "IR + Src inhibitor": ["B09", "C09", "D09"],
    },
}

PLATE_ALIASES = {
    "norm": "norm",
    "normoxia": "norm",
    "ir": "IR",
    "i-r": "IR",
}

MONTAGE_LABELS = {
    "show_condition": True,
    "show_wells": True,
    "show_timepoints": True,
    "condition_names": {},   # Example: {"normoxia": "Normoxia"}
    "well_names": {},        # Example: {"B02": "Replicate 1"}
    "timepoint_names": {},   # Example: {"T0": "Baseline"}
    "fontsize": 10,
}


def _as_list(value):
    if value is None:
        return None
    if isinstance(value, str):
        return [value]
    return list(value)


def _canonical_plate(plate):
    key = str(plate).strip().lower()
    if key not in PLATE_ALIASES:
        raise ValueError(f"Unknown plate {plate!r}. Choose from: norm, IR.")
    return PLATE_ALIASES[key]


def _resolve_conditions(plate, conditions):
    available = PLATE_CONDITIONS[plate]
    requested = _as_list(conditions)
    if requested is None:
        return list(available)

    lookup = {name.casefold(): name for name in available}
    resolved = []
    for condition in requested:
        key = str(condition).strip().casefold()
        if key not in lookup:
            raise ValueError(
                f"Unknown condition {condition!r} for plate {plate}. "
                f"Choose from: {list(available)}"
            )
        canonical = lookup[key]
        if canonical not in resolved:
            resolved.append(canonical)
    return resolved


def _discover_timepoints(labels):
    discovered = {
        match.group(1).upper()
        for label in labels
        if (match := re.search(r"_(T\d+)_", label, flags=re.IGNORECASE))
    }
    if not discovered:
        raise ValueError("No timepoints matching T<number> were found in the TIFF labels.")
    return sorted(discovered, key=lambda timepoint: int(timepoint[1:]))


def _resolve_timepoints(timepoints, available_timepoints):
    requested = _as_list(timepoints)
    if requested is None:
        return list(available_timepoints)

    resolved = []
    for timepoint in requested:
        text = str(timepoint).strip().upper()
        if text.isdigit():
            text = f"T{text}"
        if not re.fullmatch(r"T\d+", text):
            raise ValueError(f"Invalid timepoint {timepoint!r}; expected T followed by a number.")
        if text not in available_timepoints:
            raise ValueError(
                f"Timepoint {timepoint!r} is not present in this TIFF. "
                f"Available timepoints: {available_timepoints}"
            )
        if text not in resolved:
            resolved.append(text)
    return resolved


def create_montage(
    path,
    plate,
    conditions=None,
    timepoints=None,
    layout="vertical",
    downsample=4,
    channel_config=CHANNEL_CONFIG,
    label_config=MONTAGE_LABELS,
    export_path=None,
    tile_inches=3,
    border_color="white",
    border_width=2,
):
    """Display merged-channel montages for selected conditions and timepoints.

    Parameters
    ----------
    path : str or Path
        ImageJ hyperstack TIFF.
    plate : {"norm", "IR"}
        Experimental plate to select.
    conditions : str, sequence of str, or None
        One or more condition names. None selects all conditions on the plate.
    timepoints : str, int, sequence, or None
        Any subset of T0-T7. None selects all timepoints.
    layout : {"vertical", "horizontal"}
        Vertical places timepoints down rows; horizontal places them across columns.
    downsample : positive int
        Spatial stride applied while reading. Four converts 2048-pixel tiles to 512 pixels.
    channel_config : dict
        Mapping from TIFF channel IDs to display names and RGB colors.
    label_config : dict or None
        External label settings. Pass None to suppress all labels.
    export_path : str, Path, or None
        Optional PNG path or output directory. Multiple conditions get separate files.
    tile_inches : positive float
        Display size of each tile in the Matplotlib figure.
    border_color : Matplotlib color
        Color used to outline and separate image tiles.
    border_width : nonnegative float
        Tile-border width in points. Set to 0 to hide borders.
    """
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)

    plate = _canonical_plate(plate)
    selected_conditions = _resolve_conditions(plate, conditions)
    layout = str(layout).strip().lower()
    if layout not in {"vertical", "horizontal"}:
        raise ValueError("layout must be either 'vertical' or 'horizontal'.")
    if not isinstance(downsample, (int, np.integer)) or downsample < 1:
        raise ValueError("downsample must be a positive integer.")
    if tile_inches <= 0:
        raise ValueError("tile_inches must be positive.")
    if border_width < 0:
        raise ValueError("border_width must be nonnegative.")

    labels_config = None if label_config is None else {**MONTAGE_LABELS, **label_config}

    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        page = tif.pages[0]
        metadata = tif.imagej_metadata or {}
        labels = metadata.get("Labels")
        saved_ranges = metadata.get("Ranges")

        if series.axes != "ZCYX":
            raise ValueError(f"Expected ZCYX axes; found {series.axes!r}.")
        if labels is None or len(labels) != series.shape[0] * series.shape[1]:
            raise ValueError("Complete ImageJ plane labels were not found.")
        if saved_ranges is None or len(saved_ranges) != 2 * series.shape[1]:
            raise ValueError("Per-channel ImageJ display ranges were not found.")
        if page.compression != 1 or len(page.dataoffsets) != 1:
            raise ValueError("This reader expects an uncompressed, contiguous ImageJ TIFF.")

        labels = np.asarray(labels).reshape(series.shape[0], series.shape[1])
        ranges = np.asarray(saved_ranges, dtype=np.float32).reshape(series.shape[1], 2)
        plane_shape = tuple(page.shape)
        plane_pixels = int(np.prod(plane_shape))
        file_dtype = np.dtype(tif.byteorder + "u2")
        plane_bytes = plane_pixels * file_dtype.itemsize
        first_plane_offset = page.dataoffsets[0]

    available_timepoints = _discover_timepoints(labels[:, 0])
    selected_timepoints = _resolve_timepoints(timepoints, available_timepoints)

    label_pattern = re.compile(r"^(IR|norm)_([BCD]\d{2})_(T\d+)_", flags=re.IGNORECASE)
    image_index = {}
    for z_index, label in enumerate(labels[:, 0]):
        match = label_pattern.match(label)
        if match:
            label_plate, label_well, label_timepoint = match.groups()
            image_index[
                (_canonical_plate(label_plate), label_well.upper(), label_timepoint.upper())
            ] = z_index

    source_ids = [label.rsplit("_", 1)[-1].removesuffix(".tif") for label in labels[0]]
    if source_ids != list(channel_config):
        raise ValueError(
            f"TIFF channels {source_ids} do not match configured channels {list(channel_config)}."
        )

    condition_wells = {
        condition: list(PLATE_CONDITIONS[plate][condition])
        for condition in selected_conditions
    }
    missing = [
        (plate, well, timepoint)
        for wells in condition_wells.values()
        for well in wells
        for timepoint in selected_timepoints
        if (plate, well, timepoint) not in image_index
    ]
    if missing:
        raise ValueError(f"Missing requested images: {missing}")

    print(f"Plate: {plate}")
    print(f"Conditions: {selected_conditions}")
    print(f"Available timepoints ({len(available_timepoints)}): {available_timepoints}")
    print(f"Selected timepoints ({len(selected_timepoints)}): {selected_timepoints}")
    print(f"Downsample: {downsample} ({plane_shape[1] // downsample} x {plane_shape[0] // downsample} tiles)")
    print("Channels:")
    for channel_index, channel_id in enumerate(source_ids):
        settings = channel_config[channel_id]
        low, high = ranges[channel_index]
        print(
            f"  {channel_id}: {settings['name']}, color={settings['color']}, "
            f"ImageJ range={low:.2f}-{high:.2f}"
        )

    sampled_shape = (
        len(range(0, plane_shape[0], downsample)),
        len(range(0, plane_shape[1], downsample)),
    )

    results = {}
    with path.open("rb") as tif_file:
        for condition in selected_conditions:
            wells = condition_wells[condition]
            if layout == "vertical":
                nrows, ncols = len(selected_timepoints), len(wells)
            else:
                nrows, ncols = len(wells), len(selected_timepoints)

            print(f"\nCondition: {condition}")
            print(f"Wells ({len(wells)}): {wells}")
            print(f"Figure layout: {nrows} rows x {ncols} columns")

            fig, axes = plt.subplots(
                nrows,
                ncols,
                figsize=(ncols * tile_inches, nrows * tile_inches),
                squeeze=False,
            )

            for well_index, well in enumerate(wells):
                for time_index, timepoint in enumerate(selected_timepoints):
                    z_index = image_index[(plate, well, timepoint)]
                    composite = np.zeros((*sampled_shape, 3), dtype=np.float32)

                    for channel_index, channel_id in enumerate(source_ids):
                        flat_index = z_index * len(source_ids) + channel_index
                        tif_file.seek(first_plane_offset + flat_index * plane_bytes)
                        plane = np.fromfile(
                            tif_file,
                            dtype=file_dtype,
                            count=plane_pixels,
                        ).reshape(plane_shape)[::downsample, ::downsample]

                        low, high = ranges[channel_index]
                        displayed = np.clip(
                            (plane.astype(np.float32) - low) / (high - low),
                            0.0,
                            1.0,
                        )
                        color = np.asarray(channel_config[channel_id]["color"], dtype=np.float32)
                        composite += displayed[..., None] * color

                    tile = (np.clip(composite, 0.0, 1.0) * 255).astype(np.uint8)
                    row, column = (
                        (time_index, well_index)
                        if layout == "vertical"
                        else (well_index, time_index)
                    )
                    axis = axes[row, column]
                    axis.imshow(tile)
                    axis.set_xticks([])
                    axis.set_yticks([])
                    for spine in axis.spines.values():
                        spine.set_visible(border_width > 0)
                        spine.set_color(border_color)
                        spine.set_linewidth(border_width)

                    if labels_config is not None:
                        fontsize = labels_config["fontsize"]
                        if layout == "vertical":
                            if labels_config["show_wells"] and time_index == 0:
                                axis.set_title(
                                    labels_config["well_names"].get(well, well),
                                    fontsize=fontsize,
                                )
                            if labels_config["show_timepoints"] and well_index == 0:
                                axis.set_ylabel(
                                    labels_config["timepoint_names"].get(timepoint, timepoint),
                                    fontsize=fontsize,
                                )
                        else:
                            if labels_config["show_timepoints"] and well_index == 0:
                                axis.set_title(
                                    labels_config["timepoint_names"].get(timepoint, timepoint),
                                    fontsize=fontsize,
                                )
                            if labels_config["show_wells"] and time_index == 0:
                                axis.set_ylabel(
                                    labels_config["well_names"].get(well, well),
                                    fontsize=fontsize,
                                )

            if labels_config is not None and labels_config["show_condition"]:
                title = labels_config["condition_names"].get(condition, condition)
                fig.suptitle(title, fontsize=labels_config["fontsize"] + 2)
                fig.tight_layout(rect=(0, 0, 1, 0.98), pad=0.2, w_pad=0.1, h_pad=0.1)
            else:
                fig.subplots_adjust(left=0, right=1, bottom=0, top=1, wspace=0.01, hspace=0.01)

            if export_path is not None:
                base = Path(export_path)
                slug = re.sub(r"[^A-Za-z0-9]+", "-", condition).strip("-")
                if base.suffix.lower() == ".png":
                    output = (
                        base
                        if len(selected_conditions) == 1
                        else base.with_name(f"{base.stem}_{slug}.png")
                    )
                else:
                    output = base / f"{plate}_{slug}.png"
                output.parent.mkdir(parents=True, exist_ok=True)
                fig.savefig(output, dpi=300, bbox_inches="tight", pad_inches=0.05)
                print(f"Exported: {output}")

            plt.show()
            results[condition] = (fig, axes)

    return results


In [ ]:
# Improve inline montage display quality
%config InlineBackend.figure_format = "retina"
plt.rcParams["figure.dpi"] = 200

# One separate figure will be displayed for each selected condition.
# Normoxia plate
norm_montages = create_montage(
    TIFF_PATH,
    plate="norm",
    conditions=None,       # None = all conditions on the selected plate
    timepoints=None,       # None = all timepoints discovered in the TIFF
    layout="vertical",    # timepoints down rows; wells across columns
    downsample=2,          # if 4 is selected, 2048 x 2048 tiles become 512 x 512 
    label_config=MONTAGE_LABELS,
    export_path=None,      
    tile_inches=4,
    border_color="white",
    border_width=2
)

In [ ]:
# Improve inline montage display quality
%config InlineBackend.figure_format = "retina"
plt.rcParams["figure.dpi"] = 200

# One separate figure will be displayed for each selected condition.
# IR plate
ir_montages = create_montage(
    TIFF_PATH,
    plate="IR",
    conditions=None,       # None = all conditions on the selected plate
    timepoints=None,       # None = all timepoints discovered in the TIFF
    layout="vertical",    # timepoints down rows; wells across columns
    downsample=2,          # if 4 is selected, 2048 x 2048 tiles become 512 x 512 
    label_config=MONTAGE_LABELS,
    export_path=None,      
    tile_inches=4,
    border_color="white",
    border_width=2
)